In [1]:
!pip install mido

     -------------------------------------- 54.6/54.6 kB 944.1 kB/s eta 0:00:00



[notice] A new release of pip available: 22.3.1 -> 25.1.1
[notice] To update, run: C:\Users\voldo\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip


In [2]:
import mido
import pandas as pd
import numpy as np



def midi_to_dataframe(midi_filepath):
    """
    Converts a MIDI file to a Pandas DataFrame representing a piano roll.

    Each row in the DataFrame corresponds to a single MIDI tick.
    Each column corresponds to a MIDI note number (0-127).
    The cell value is the velocity of the note if it's 'on' at that tick, 
    otherwise 0.

    Args:
        midi_filepath (str): Path to the MIDI file.

    Returns:
        pd.DataFrame: DataFrame representing the piano roll, or None if error.
    """
    try:
        mid = mido.MidiFile(midi_filepath)
    except Exception as e:
        print(f"Error opening MIDI file {midi_filepath}: {e}")
        return None

    all_events = []
    for i, track in enumerate(mid.tracks):
        current_abs_tick_track = 0
        for msg in track:
            current_abs_tick_track += msg.time  # Add delta time to get absolute time for this event in the track
            if msg.type in ['note_on', 'note_off']:
                all_events.append({
                    'absolute_tick': current_abs_tick_track,
                    'type': msg.type,
                    'note': msg.note,
                    'velocity': msg.velocity
                })
            # You could also capture other messages like 'set_tempo' if needed for other analyses

    if not all_events:
        print(f"No note_on/note_off events found in {midi_filepath}.")
        # Return an empty DataFrame with appropriate columns if no events
        return pd.DataFrame(np.zeros((0, 128), dtype=np.uint8), columns=range(128))

    # Sort all events by their absolute tick time
    all_events.sort(key=lambda x: x['absolute_tick'])

    # Determine the total number of ticks for the DataFrame
    # This will be the tick of the last event + 1 (or more if the last event has duration)
    # For simplicity in this piano roll, we just go up to the last event's start time.
    # A more precise total_ticks would consider the duration of the last note.
    # However, our filling method handles this by propagating velocities.
    total_ticks = 0
    if all_events:
        total_ticks = all_events[-1]['absolute_tick'] + 1


    # Initialize the piano roll data array (ticks x 128 notes)
    # Using uint8 as velocity is 0-127
    piano_roll_data = np.zeros((total_ticks, 128), dtype=np.uint8)

    # Keeps track of the current velocity for each note
    current_velocities = np.zeros(128, dtype=np.uint8)
    
    event_idx = 0
    for tick_idx in range(total_ticks):
        # Process all MIDI events that occur at the current tick_idx
        while event_idx < len(all_events) and all_events[event_idx]['absolute_tick'] == tick_idx:
            event = all_events[event_idx]
            note = event['note']
            velocity = event['velocity']
            
            if event['type'] == 'note_on' and velocity > 0:
                current_velocities[note] = velocity
            elif event['type'] == 'note_off' or (event['type'] == 'note_on' and velocity == 0):
                # A note_on with velocity 0 is often treated as a note_off
                current_velocities[note] = 0
            
            event_idx += 1
        
        # Assign the current state of all notes (their velocities) to this tick in the piano roll
        piano_roll_data[tick_idx, :] = current_velocities

    # Create Pandas DataFrame
    df_piano_roll = pd.DataFrame(piano_roll_data, columns=range(128))
    df_piano_roll.index.name = 'Tick'
    df_piano_roll.columns.name = 'MIDI Note Number'
    
    return df_piano_roll



# --- Usage Example ---
if __name__ == '__main__':
    # Replace 'your_midi_file.mid' with the actual path to your MIDI file
    midi_file_path = "Queen - Bohemian Rhapsody.mid"  # Make sure this file exists
    
    # To test, you might need a simple MIDI file. 
    # You can create one with music software or find one online.
    # For a quick test, you can try to create a dummy file if you have mido installed
    # (this is just for testing the function if you don't have a file handy)
    try:
        # Create a simple dummy MIDI file for testing
        mid_test = mido.MidiFile()
        track_test = mido.MidiTrack()
        mid_test.tracks.append(track_test)
        track_test.append(mido.Message('note_on', note=60, velocity=100, time=0))      # C4 on at tick 0
        track_test.append(mido.Message('note_on', note=64, velocity=90, time=100))     # E4 on at tick 100
        track_test.append(mido.Message('note_off', note=60, velocity=0, time=100))     # C4 off at tick 200 (100+100)
        track_test.append(mido.Message('note_off', note=64, velocity=0, time=100))     # E4 off at tick 300 (200+100)
        mid_test.save('test_dummy.mid')
        midi_file_path = 'test_dummy.mid' # Use this dummy file
        print(f"Using dummy MIDI file: {midi_file_path}")
    except Exception as e:
        print(f"Could not create dummy MIDI file for testing: {e}")
        print("Please provide a valid path to a MIDI file for 'midi_file_path'")


    if midi_file_path == 'your_midi_file.mid' and midi_file_path not in ['test_dummy.mid']: # check if user still has placeholder
         print(f"Please replace 'your_midi_file.mid' with an actual MIDI file path.")
    else:
        df_midi = midi_to_dataframe(midi_file_path)

        if df_midi is not None:
            print("\nMIDI Piano Roll DataFrame:")
            # Displaying info or head as the full DataFrame can be very large
            print(df_midi.info())
            if not df_midi.empty:
                print("\nDataFrame Head (first 5 ticks):")
                print(df_midi.head())
                print("\nDataFrame Tail (last 5 ticks):")
                print(df_midi.tail())

                # Example: Check value for a specific note (e.g., note 60) if it exists in columns
                if 60 in df_midi.columns:
                    print("\nVelocities for Note 60 (Middle C) over time (first 10 ticks if present):")
                    print(df_midi[60].head(10))
            else:
                print("The DataFrame is empty (no note events or zero total ticks).")

Using dummy MIDI file: test_dummy.mid

MIDI Piano Roll DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Columns: 128 entries, 0 to 127
dtypes: uint8(128)
memory usage: 37.8 KB
None

DataFrame Head (first 5 ticks):
MIDI Note Number  0    1    2    3    4    5    6    7    8    9    ...  118  \
Tick                                                                ...        
0                   0    0    0    0    0    0    0    0    0    0  ...    0   
1                   0    0    0    0    0    0    0    0    0    0  ...    0   
2                   0    0    0    0    0    0    0    0    0    0  ...    0   
3                   0    0    0    0    0    0    0    0    0    0  ...    0   
4                   0    0    0    0    0    0    0    0    0    0  ...    0   

MIDI Note Number  119  120  121  122  123  124  125  126  127  
Tick                                                           
0                   0    0    0    0    0    0    0    0    0  

In [6]:
pd.set_option("display.max_rows", 1000)

df_midi = midi_to_dataframe("Queen - Bohemian Rhapsody.mid")

In [10]:
df_midi[[10, 11, 12, 13, 14, 15, 16, 17, 18]].head(1000)

MIDI Note Number,10,11,12,13,14,15,16,17,18
Tick,,,,,,,,,
0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0


In [11]:
df_midi.tail(1000)

MIDI Note Number,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
Tick,,,,,,,,,,,,,,,,,,,,,
91641,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
91642,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
91643,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
91644,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
91645,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
91646,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
91647,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
91648,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
91649,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
is_row_all_zeros = df_midi.eq(0).all(axis=1)

# We want to keep rows where NOT all values are zeros.
filtered_df = df_midi[~is_row_all_zeros]

In [13]:
filtered_df.head(1000)

MIDI Note Number,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
Tick,,,,,,,,,,,,,,,,,,,,,
672,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
673,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
674,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
675,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
676,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
677,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
678,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
679,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
680,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
